# 🧠 Module 08: Modern LLMs, Alignment & Reasoning
### *From InstructGPT (2022) and DPO (2023) to DeepSeek MoE, GRPO (2024/2025), and Test-Time Reasoning*

Welcome to **Module 08 of Project Synapse**. This module dissects the modern Large Language Model architecture and post-training stack:
1. **Conditional Computation**: Sparse Mixture-of-Experts (MoE) with load-balancing auxiliary loss.
2. **Alignment without Reward Models**: Direct Preference Optimization (DPO).
3. **Critic-Free Reinforcement Learning**: Group Relative Policy Optimization (GRPO) from DeepSeekMath / DeepSeek-R1.
4. **Test-Time Compute**: Scaling reasoning capabilities via inference search and verification.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'python' / 'module08_llm_reasoning_and_alignment'))

import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from moe_routing_scratch import TopKRouter, SparseMoELayer
from dpo_and_grpo_loss import DPOLoss, GRPOLoss, TestTimeReasoningSimulator

print("All Module 08 dependencies loaded successfully!")

## 1. Sparse Mixture of Experts (MoE) & Router Dynamics

In dense transformers, every parameter is activated for every token. In Sparse MoE, a gating network routes each token to only the Top-$ most relevant experts:
17137\text{logits} = x W_g, \quad \text{TopK} = \text{argsort}(\text{logits})[-K:], \quad w = \text{softmax}(\text{TopK})17137

### The Load Balancing Auxiliary Loss
Without an auxiliary loss, the router suffers from **router collapse**—a vicious cycle where 1 or 2 experts receive all tokens, while other experts remain dead.
The auxiliary loss balances dispatch:
17137\mathcal{L}_{\text{balance}} = \alpha \cdot N \sum_{i=1}^N f_i P_i17137
where $ is the empirical fraction of tokens dispatched to expert $, and $ is the average routing probability.

In [ ]:
# Instantiate MoE with 8 routed experts and Top-2 routing
d_model = 64
d_ff = 128
num_experts = 8
top_k = 2

moe = SparseMoELayer(d_model=d_model, d_ff=d_ff, num_routed_experts=num_experts, top_k=top_k, num_shared_experts=1)

# Synthetic token batch: 32 tokens
tokens = torch.randn(32, d_model)
out, aux_loss, diag = moe(tokens)

# Visualize Expert Utilization Histogram
plt.figure(figsize=(9, 4), dpi=100)
expert_fractions = diag['expert_dispatch_fraction'].numpy()
bars = plt.bar(range(num_experts), expert_fractions, color='#8b5cf6', edgecolor='#4c1d95', width=0.6)
plt.axhline(top_k / num_experts, color='#ef4444', linestyle='--', label=f'Ideal Uniform Dispatch ({top_k / num_experts:.1%})')
plt.title(f'MoE Expert Dispatch Distribution (Top-{top_k} of {num_experts} Experts)', fontsize=12)
plt.xlabel('Expert Index')
plt.ylabel('Fraction of Tokens Routed $')
plt.grid(True, alpha=0.3, axis='y')
plt.legend()
plt.tight_layout()
plt.show()

## 2. Direct Preference Optimization (DPO)

Instead of the 3-step RLHF pipeline (SFT -> Train Reward Model -> PPO Policy), DPO directly optimizes the policy using the exact closed-form solution to the Bradley-Terry preference objective:
17137\mathcal{L}_{\text{DPO}}(\pi_\theta; \pi_{\text{ref}}) = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)} \right) \right]17137

In [ ]:
# Plot DPO Loss Surface as a function of the Reward Margin
margins = np.linspace(-4, 4, 200)
betas = [0.05, 0.1, 0.2, 0.5]

plt.figure(figsize=(10, 4.5), dpi=100)
for b in betas:
    # Loss = -log(sigmoid(b * margin))
    loss_vals = -np.log(1.0 / (1.0 + np.exp(-b * margins)))
    plt.plot(margins, loss_vals, label=f'Beta = {b}', lw=2)

plt.axvline(0, color='gray', linestyle=':', label='Zero Margin')
plt.title('DPO Loss vs Implicit Reward Margin $\Delta r = r_\theta(y_w) - r_\theta(y_l)$', fontsize=12)
plt.xlabel('Implicit Reward Margin $(r_w - r_l) / \beta$')
plt.ylabel('DPO Objective Loss')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 3. Group Relative Policy Optimization (GRPO)

Introduced in DeepSeekMath and DeepSeek-R1, GRPO removes the critic/value network entirely.
For each prompt $, sample $ candidate reasoning paths $\{o_1, \dots, o_G\}$, compute rewards $, and calculate group relative advantages:
17137A_i = \frac{r_i - \text{mean}(\mathbf{r})}{\text{std}(\mathbf{r}) + \epsilon}17137
The surrogate objective:
17137\mathcal{L}_{\text{GRPO}}(\theta) = -\frac{1}{G} \sum_{i=1}^G \min\left( \frac{\pi_\theta(o_i)}{\pi_{\text{old}}(o_i)} A_i, \text{clip}\left(\frac{\pi_\theta(o_i)}{\pi_{\text{old}}(o_i)}, 1-\epsilon, 1+\epsilon\right) A_i \right) + \beta_{\text{KL}} \mathbb{D}_{\text{KL}}(\pi_\theta \| \pi_{\text{ref}})17137

In [ ]:
# Demonstrate GRPO Advantage Calculation & Gradient Flow
grpo = GRPOLoss(clip_epsilon=0.2, kl_weight=0.04)

# Group of 6 rollouts: [Math Correct, Math Correct, Partial, Wrong, Wrong, Wrong]
rewards = torch.tensor([1.0, 1.0, 0.5, 0.0, 0.0, 0.0])
policy_logp = torch.tensor([-1.2, -1.1, -1.8, -2.5, -2.8, -3.0], requires_grad=True)
old_logp = torch.tensor([-1.3, -1.2, -1.7, -2.4, -2.7, -2.9])
ref_logp = torch.tensor([-1.4, -1.3, -1.8, -2.3, -2.6, -2.8])

loss_grpo, metrics_grpo = grpo(policy_logp, old_logp, ref_logp, rewards)
loss_grpo.backward()

# Plot Group Rewards vs Normalized Advantages
fig, ax1 = plt.subplots(figsize=(10, 4), dpi=100)
colors = ['#10b981' if r >= 0.5 else '#ef4444' for r in rewards.numpy()]

ax1.bar(np.arange(6) - 0.2, rewards.numpy(), width=0.4, label='Verifier Reward $', color=colors, alpha=0.7)
ax1.bar(np.arange(6) + 0.2, metrics_grpo['advantages'].numpy(), width=0.4, label='Group Advantage $', color='#3b82f6', alpha=0.7)

ax1.set_xticks(range(6))
ax1.set_xticklabels([f'Rollout {i+1}' for i in range(6)])
ax1.set_title('GRPO: Raw Rewards vs Normalized Group Advantages', fontsize=12)
ax1.axhline(0, color='gray', linestyle='--')
ax1.grid(True, alpha=0.2, axis='y')
ax1.legend()
plt.tight_layout()
plt.show()

## 4. Test-Time Compute & Reasoning Search Tree

Scaling test-time compute allows LLMs to explore multiple hypotheses, self-correct errors, and reach verified solutions (e.g. OpenAI o1/o3, DeepSeek-R1).

In [ ]:
# Run Test-Time Compute Reasoning Search Simulator
sim = TestTimeReasoningSimulator()
tree = sim.build_toy_search_tree()

print("Verified Chain of Thought:")
for i, step in enumerate(tree['best_path']):
    print(f"  Step {i+1}: {step}")